# SCADA Flow Data Ingestion Pipeline

**mcom technology** | SQL Server → Databricks Delta Lake

## Pipeline Architecture

```
SQL Server (on-prem)  →  pymssql  →  pandas chunks (500K rows)  →  local Parquet
     ↓                                                                  ↓
SCADA_FLOW_2025                                              4 parallel threads
SCADA_FLOW_2026                                                        ↓
                                                            UC Volume (staging)
                                                                       ↓
                                                          read_files() → Delta
```

## Source: SQL Server (`LAPTOP-954ODF5J\SQLEXPRESS`)
## Target: `workspace.scada_flow_2025` and `workspace.scada_flow_2026` schemas

## Tables Discovered & Loaded

| Database | Table | Rows | Status |
|---|---|---|---|
| SCADA_FLOW_2025 | tb_logedata | 27,279,940 | OK |
| SCADA_FLOW_2026 | tb_device_list | 608 | OK |
| SCADA_FLOW_2026 | tb_device_list_parameter | 18,240 | OK |
| SCADA_FLOW_2026 | tb_livedata | 1,537 | OK |
| SCADA_FLOW_2026 | tb_livedata_last | 405 | OK |
| SCADA_FLOW_2026 | tb_mobile_apps | 71 | OK |
| SCADA_FLOW_2026 | tb_report_alert | 67,124 | OK |
| SCADA_FLOW_2026 | tb_report_totalizer | 162,134 | OK |
| SCADA_FLOW_2026 | tb_userlogin | 385 | OK |
| **Total** | **9 tables** | **27,530,444** | **All OK** |

## Pipeline Steps

**Step 1:** pymssql connects to SQL Server, discovers non-empty tables via INFORMATION_SCHEMA

**Step 2:** Reads each table in 500K-row chunks via `pd.read_sql()`, writes as local Parquet via pyarrow

**Step 3:** Uploads Parquet files to UC staging volumes using 4 parallel threads (`databricks-sdk`)

**Step 4:** Creates Delta tables via `read_files()` SQL function

**Step 5:** Verifies row counts match between source and Delta

## Performance Notes
- v1 (INSERT VALUES) was abandoned for the 27.2M-row table - too slow
- v2 (Parquet + parallel upload + read_files) is orders of magnitude faster
- Serverless compute cannot reach on-prem SQL Server directly
- Solution: run Python script locally, push data to Databricks via SDK

---
*© 2026 mcom technology. All rights reserved.*